# Encuentro 3 · NLP Lab: de una frase a una predicción

**Notebook práctico asociado a la aplicación Streamlit v2**  
**Tema:** representación de texto, tokenización WordPiece y clasificación de sentimientos con Hugging Face.

**Pregunta guía:** ¿Cómo pasa una frase de texto a tokens, identificadores numéricos y cinco puntuaciones de sentimiento?

Este notebook es **autónomo**: funciona sin descargar `app.py` ni `nlp_core.py`. Al final verás cómo se trasladan estas funciones al repositorio de GitHub.

**Resultados del ejercicio:**
1. Ver los tokens y lemas de spaCy.
2. Identificar los tokens reales y los ID usados por el clasificador.
3. Obtener las puntuaciones de las **cinco** categorías (1–5 estrellas).
4. Visualizar un gráfico y comparar frases con negación.
5. Entender qué código corresponde al notebook, `nlp_core.py` y `app.py`.

> **Nota:** el modelo clasifica sentimientos en una escala de estrellas basada en reseñas de productos; no es un clasificador de motivos de solicitudes estudiantiles. No se entrena un modelo nuevo.

## 0. Preparación del entorno

Requiere **Python 3**, conexión a Internet en la primera ejecución y memoria suficiente para cargar el modelo. Puedes ejecutarlo en **Google Colab** o Jupyter. La instalación siguiente está pensada para un entorno de notebook; Streamlit Cloud utiliza su propio `requirements.txt`.

La primera celda puede tardar porque instala las librerías y descarga el modelo lingüístico español de spaCy. **Reinicia el kernel si el entorno lo solicita.**

In [ ]:
%pip -q install "spacy>=3.8.0,<3.9.0" "transformers==4.57.6" "pandas>=2.2.0,<3.0" "matplotlib>=3.8.0"


In [ ]:
import sys
import subprocess

# Descarga el mismo modelo lingüístico que usa la aplicación.
subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_sm", "-q"])


In [ ]:
import re
import math
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from transformers import pipeline
from IPython.display import display

MODELO_SENTIMIENTO = "nlptown/bert-base-multilingual-uncased-sentiment"
MAX_TOKENS = 128
print("Modelo de sentimiento:", MODELO_SENTIMIENTO)


## 1. Escribe la frase

Modifica `frase` y ejecuta de nuevo las celdas para analizar un texto diferente. El texto **no se limpia** antes de enviarse al clasificador: conservar negaciones como «no» es importante.

In [ ]:
frase = "Me gustan mucho las clases con el profesor Juan Felipe"

if not frase.strip():
    raise ValueError("Debes escribir una frase no vacía.")

print("Texto:", frase)
print("Separación simple por espacios:", frase.split())


## 2. spaCy: palabras, lemas y stop words

En el notebook original, spaCy se usa para estudiar cómo el procesamiento lingüístico divide el texto. **Los tokens de spaCy no son los tokens enviados al clasificador**: se muestran como comparación educativa.

In [ ]:
nlp = spacy.load("es_core_news_sm")
doc = nlp(frase)
tabla_spacy = pd.DataFrame([
    {
        "Token": token.text,
        "Lema": token.lemma_,
        "Stop word": bool(token.is_stop),
        "Puntuación": bool(token.is_punct),
    }
    for token in doc
])
print(f"spaCy encontró {len(doc)} tokens")
display(tabla_spacy)


In [ ]:
def limpiar(texto):
    """Demostración del notebook original; NO se utiliza para la predicción."""
    return " ".join(
        token.lemma_.lower()
        for token in nlp(texto)
        if not token.is_stop and not token.is_punct
    )

for ejemplo in ("La clase de hoy me gustó mucho", "No me gustó la clase de hoy"):
    print(repr(ejemplo), "→", repr(limpiar(ejemplo)))


**Para discutir:** ¿se perdió la negación al eliminar *stop words*? El preprocesamiento no es neutro; la limpieza anterior es solo una demostración. La aplicación envía el texto original al clasificador.

## 3. Cargar el modelo preentrenado

Usaremos **el mismo** modelo de la aplicación Streamlit v2. El `pipeline` incluye tanto el tokenizador como la red neuronal.

- **Tokenizador del clasificador:** convierte el texto en subpalabras y `input_ids`.
- **Modelo Transformer:** procesa internamente esos ID y obtiene cinco puntuaciones.

El tokenizador de **BETO** usado en otro nivel del notebook original es diferente; no se utiliza aquí porque el objetivo es mostrar los tokens realmente utilizados para esta predicción.

In [ ]:
# device=-1 fuerza ejecución en CPU, equivalente a la app en Streamlit Cloud.
clasificador = pipeline(
    task="text-classification",
    model=MODELO_SENTIMIENTO,
    device=-1,
)
tokenizador = clasificador.tokenizer
print("Etiquetas de salida:", clasificador.model.config.id2label)


## 4. WordPiece: tokens e identificadores del modelo

El tokenizador puede dividir palabras en subpalabras, por ejemplo `gus` + `##ta`. Los tokens especiales `[CLS]` y `[SEP]` delimitan la secuencia. Sus ID son índices de vocabulario, **no** valores de sentimiento.

In [ ]:
codificacion = tokenizador(
    frase,
    truncation=True,
    max_length=MAX_TOKENS,
    add_special_tokens=True,
)
ids = codificacion["input_ids"]
tokens = tokenizador.convert_ids_to_tokens(ids)

tabla_tokens = pd.DataFrame({
    "Posición": range(len(ids)),
    "Token del modelo": tokens,
    "ID": ids,
})

print("Tokens utilizados:", tokens)
print("Número de tokens (incluidos los especiales):", len(ids))
display(tabla_tokens)


## 5. Predicción: obtener las **cinco** puntuaciones

En el notebook original se llama a `pipeline(...)` y se observa solo una etiqueta. La aplicación v2 utiliza `top_k=None` para recuperar las cinco y mostrar cómo se distribuyen. Esto no convierte el modelo en un sistema de certeza calibrada.

In [ ]:
resultados_crudos = clasificador(
    frase,
    truncation=True,
    max_length=MAX_TOKENS,
    top_k=None,
)
print("Respuesta cruda del clasificador:")
print(resultados_crudos)


In [ ]:
# Mismo procesamiento de etiquetas que usa nlp_core.py de la app v2.
PATRON_ETIQUETA = re.compile(r"([1-5]) stars?", re.IGNORECASE)

def procesar_puntuaciones(predicciones):
    if (isinstance(predicciones, list) and len(predicciones) == 1
            and isinstance(predicciones[0], list)):
        predicciones = predicciones[0]
    if not isinstance(predicciones, list):
        raise ValueError("La respuesta del modelo no es una lista.")

    valores = {}
    for pred in predicciones:
        etiqueta = str(pred["label"]).strip()
        match = PATRON_ETIQUETA.fullmatch(etiqueta)
        if not match:
            raise ValueError(f"Etiqueta no reconocida: {etiqueta!r}")
        estrellas = int(match.group(1))
        puntuacion = float(pred["score"])
        if estrellas in valores or not math.isfinite(puntuacion) or not (0 <= puntuacion <= 1):
            raise ValueError("Puntuaciones duplicadas o inválidas.")
        valores[estrellas] = puntuacion

    if set(valores) != set(range(1, 6)):
        raise ValueError("Faltan puntuaciones de alguna etiqueta.")
    return [
        {
            "estrellas": n,
            "etiqueta": f"{n} star" if n == 1 else f"{n} stars",
            "puntuacion": valores[n],
        }
        for n in range(1, 6)
    ]


distribucion = procesar_puntuaciones(resultados_crudos)
ganadora = max(distribucion, key=lambda x: x["puntuacion"])
tabla_puntajes = pd.DataFrame(distribucion)
tabla_puntajes["porcentaje"] = 100 * tabla_puntajes["puntuacion"]

print(f"Etiqueta predicha: {ganadora['etiqueta']}")
print(f"Puntuación de la etiqueta: {ganadora['puntuacion']:.1%}")
print(f"Suma de las cinco puntuaciones: {sum(x['puntuacion'] for x in distribucion):.5f}")
display(tabla_puntajes)


## 6. Gráfico de las cinco estrellas

La barra mayor señala la etiqueta seleccionada por el modelo. **No** representa la proporción de palabras positivas y no garantiza que la predicción sea correcta.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(tabla_puntajes["etiqueta"], tabla_puntajes["porcentaje"])
ax.set_xlabel("Puntuación del modelo (%)")
ax.set_ylabel("Etiqueta")
ax.set_title("Distribución de puntuaciones para la frase")
ax.set_xlim(0, max(100, tabla_puntajes["porcentaje"].max() * 1.1))
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 7. Integrar todo en una función: `analizar_frase()`

En el notebook original había una función `cadena()` que reunía el análisis. En Streamlit v2 se ha transformado en `analizar_frase()` para devolver un diccionario estructurado. El siguiente código reproduce esa lógica, de modo que puedas ver su funcionamiento **sin Streamlit**.

In [ ]:
def analizar_frase(texto, nlp, clasificador, max_length=128):
    texto = texto.strip()
    if not texto:
        raise ValueError("Debes ingresar una frase no vacía.")
    if max_length < 3:
        raise ValueError("max_length debe ser al menos 3.")

    tokens_spacy = [
        {
            "Token": token.text,
            "Lema": token.lemma_,
            "Stop word": bool(token.is_stop),
            "Puntuación": bool(token.is_punct),
        }
        for token in nlp(texto)
    ]

    tok = clasificador.tokenizer
    ids_originales = tok(texto, truncation=False, add_special_tokens=True)["input_ids"]
    ids_usados = tok(
        texto, truncation=True, max_length=max_length, add_special_tokens=True
    )["input_ids"]
    tokens_modelo = tok.convert_ids_to_tokens(ids_usados)
    tabla_tokens = [
        {"Posición": i, "Token del modelo": token, "ID": int(token_id)}
        for i, (token, token_id) in enumerate(zip(tokens_modelo, ids_usados))
    ]

    predicciones = clasificador(
        texto, truncation=True, max_length=max_length, top_k=None
    )
    distribucion = procesar_puntuaciones(predicciones)
    ganadora = max(distribucion, key=lambda item: item["puntuacion"])

    return {
        "texto": texto,
        "tokens_spacy": tokens_spacy,
        "tokens_modelo": tabla_tokens,
        "total_spacy": len(tokens_spacy),
        "total_modelo": len(ids_usados),
        "total_original_modelo": len(ids_originales),
        "truncado": len(ids_originales) > max_length,
        "etiqueta": ganadora["etiqueta"],
        "puntuacion": ganadora["puntuacion"],
        "distribucion": distribucion,
    }

salida = analizar_frase(frase, nlp, clasificador, MAX_TOKENS)
print("Resultado integrado:", salida["etiqueta"], f'({salida["puntuacion"]:.1%})')
display(pd.DataFrame(salida["tokens_modelo"]))
display(pd.DataFrame(salida["distribucion"]))


## 8. Experimento: cambiar una negación

Las siguientes frases difieren por la palabra «No». El objetivo es observar **cómo cambian** los cinco puntajes, sin asumir de antemano el resultado del modelo.

In [ ]:
frases_prueba = [
    "La clase de hoy me gustó mucho",
    "No me gustó la clase de hoy",
]

filas = []
for texto in frases_prueba:
    resultado = analizar_frase(texto, nlp, clasificador, MAX_TOKENS)
    fila = {"Frase": texto, "Predicción": resultado["etiqueta"]}
    fila.update({f"{d['estrellas']} ★": round(d["puntuacion"] * 100, 2)
                 for d in resultado["distribucion"]})
    filas.append(fila)
display(pd.DataFrame(filas))


## 9. Experimento: truncamiento

Al limitar artificialmente los tokens, el modelo puede perder información al final. En la aplicación real se usa `MAX_TOKENS = 128`; aquí se emplea un límite menor **solo para demostrar el problema**.

In [ ]:
frase_larga = (
    "La plataforma falló durante el taller y la clase se retrasó, "
    "pero finalmente el profesor resolvió todas las dudas y me gustó la sesión."
)
resultado_largo = analizar_frase(frase_larga, nlp, clasificador, max_length=12)
print("Tokens originales:", resultado_largo["total_original_modelo"])
print("Tokens usados:", resultado_largo["total_modelo"])
print("¿Hubo truncamiento?:", resultado_largo["truncado"])
print("Tokens conservados:", [fila["Token del modelo"] for fila in resultado_largo["tokens_modelo"]])
print("Etiqueta con límite artificial:", resultado_largo["etiqueta"])


## 10. ¿Cómo se convierte este notebook en la aplicación de GitHub?

| Elemento | Notebook | Aplicación de Streamlit |
|---|---|---|
| Escribir la frase | Variable `frase` | `st.text_area(...)` |
| spaCy | `nlp(frase)` | `cargar_spacy()` y `analizar_frase()` |
| Modelo Hugging Face | `pipeline(...)` | `cargar_clasificador()` |
| Tokens + IDs | `clasificador.tokenizer(...)` | `nlp_core.py` |
| Cinco puntuaciones | `clasificador(..., top_k=None)` | `nlp_core.py` |
| Gráfico | `matplotlib` | `barras_puntuaciones()` en `ui_components.py` |
| Mostrar salida | `print()` / `display()` | `tarjeta_resultado()`, `fichas_tokens()` y `st.dataframe()` |

**Estructura del repositorio:**

```text
NLP_LAB/
├── .streamlit/
│   └── config.toml         # Tema oscuro con la paleta de seis colores
├── tests/
│   ├── test_nlp_core.py
│   └── test_ui_components.py
├── app.py                  # Interfaz, formulario y secciones de resultados
├── nlp_core.py             # analizar_frase() y procesar_puntuaciones()
├── ui_components.py        # Tarjetas, barras de puntuación y fichas de tokens
├── styles.css              # Colores, tarjetas y diseño responsivo
├── requirements.txt        # Dependencias para Streamlit Cloud
├── README.md
├── NLP_Lab.ipynb           # Este notebook (material didáctico)
└── .gitignore
```

Streamlit no ejecuta automáticamente todas las celdas `.ipynb`. En Community Cloud el archivo principal es **`app.py`**. `nlp_core.py` contiene las funciones que aquí has probado una a una. Para desplegar no es necesario ejecutar el notebook, pero sí puedes conservarlo en el repositorio como documentación pedagógica.

### Fragmento de interfaz en `app.py` (ilustrativo)

```python
with st.form("formulario_frase"):
    frase = st.text_area("Frase para analizar")
    analizar = st.form_submit_button("Mostrar tokens y predicción")

if analizar and frase.strip():
    salida = analizar_frase(frase, cargar_spacy(), cargar_clasificador())
    st.metric("Etiqueta", salida["etiqueta"])
    st.dataframe(salida["tokens_modelo"])
    st.bar_chart(pd.DataFrame(salida["distribucion"]), x="etiqueta", y="puntuacion")
```

> Es una versión simplificada. La app real reemplaza `st.metric()` y `st.bar_chart()` por las tarjetas y barras HTML de `ui_components.py`, aplica el diseño de `styles.css` e incluye las precauciones de carga de modelos de `app.py`.

## 11. Actividad de cierre

1. Modifica la variable `frase` para ingresar una frase neutra. ¿Qué estrellas recibe? ¿La salida significa que el modelo haya comprendido el motivo de la consulta?
2. Examina `tokens_modelo`: identifica `[CLS]`, `[SEP]` y al menos una subpalabra `##` (si aparece).
3. Compara las cinco puntuaciones de dos frases con negación y sin negación. ¿Qué cambió?
4. Evalúa cómo influye el truncamiento en una frase larga y qué información se descarta.
5. Explica en dos oraciones por qué **tokenizar no es predecir**.

**Interpretación responsable:** las cinco puntuaciones de clasificación suelen sumar ~100 %, pero no equivalen a precisión, fiabilidad calibrada ni probabilidad real de que una opinión educativa sea correcta o incorrecta.

**Fuente de la actividad:** adaptación de `Encuentro_3_Laboratorio_version_compacta.ipynb` y del proyecto `nlp_lab_frase_streamlit_v2.zip`. Se conserva el clasificador original, y el nuevo material agrega la salida de las cinco estrellas y una visualización.